# Week 4 — Bayesian Inference, Priors and Posterior Prediction

**Contact:** 3 hours lecture  
**Course:** MSc Astrostatistics for the Survey Era

## Learning notebooks from the Crete school
- `../../06_Bayesian/Bayesian_1.ipynb`
- `../../06_Bayesian/Bayesian_2_Bayesball.ipynb`
- `../../06_Bayesian/Bayesian_3.ipynb`

These Crete notebooks are **learning material**. This lecture notebook supplies the semester narrative, deeper astronomical interpretation, and the binary-SMBH case study.

## Three-hour pacing
- **0:00–0:20** — recap, motivating scientific question, diagnostic poll
- **0:20–1:05** — theory block I + worked derivation
- **1:05–1:15** — short break / discussion
- **1:15–2:00** — theory block II + computational demonstration
- **2:00–2:10** — short break
- **2:10–2:45** — binary-SMBH case study / failure mode
- **2:45–3:00** — synthesis, exit questions, bridge to the 4-hour practical

## Learning objectives

- Interpret Bayes' theorem as a model for updating uncertainty.
- Distinguish likelihood, prior, posterior, posterior predictive distribution and evidence.
- Use prior predictive simulation to diagnose implausible priors.
- Perform sensitivity analysis to reasonable prior choices.
- Use posterior predictive checks to assess whether a model reproduces the observed data.

## 1. Bayes' theorem as an inference engine


Bayesian inference combines a likelihood with prior information:

\[
p(\theta\mid D)=\frac{p(D\mid\theta)p(\theta)}{p(D)},
\qquad
p(D)=\int p(D\mid\theta)p(\theta)\,d\theta.
\]

The likelihood and posterior answer different questions. \(p(D\mid\theta)\) is a model for data given a parameter; \(p(\theta\mid D)\) is uncertainty about the parameter after observing the data.

The prior is part of the model. It can encode physical constraints, previous knowledge or deliberately weak information, but "flat" is not synonymous with "uninformative": a prior flat in \(M\) is not flat in \(\log M\).

> **Discussion checkpoint.** Identify the estimand, observed data, and strongest modelling assumption in the example just discussed.

### Worked computational demonstration — Grid posterior for log-mass mean

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
import numpy as np, sys, matplotlib.pyplot as plt
sys.path.append('../src')
from course_utils import load_scalar_sample
df=load_scalar_sample()
y=np.log10(df['inputs__total_mass'].to_numpy())
sigma=np.std(y,ddof=1)

mu_grid=np.linspace(y.mean()-.3,y.mean()+.3,1000)
loglike=np.array([-0.5*np.sum(((y-m)/sigma)**2) for m in mu_grid])
# weakly informative Normal prior
mu0, tau = 8.0, 2.0
logprior=-0.5*((mu_grid-mu0)/tau)**2
logpost=loglike+logprior
post=np.exp(logpost-logpost.max())
post/=np.trapezoid(post,mu_grid)
plt.plot(mu_grid,post)
plt.xlabel(r'$\mu_{\log M}$')
plt.ylabel('posterior density')
plt.show()

## 2. Prior predictive checking


Before observing the data, the model implies a prior predictive distribution

\[
p(\tilde D)=\int p(\tilde D\mid\theta)p(\theta)\,d\theta.
\]

Draw from the prior, simulate data and ask whether the resulting observations are physically plausible. A prior that routinely predicts impossible luminosities, absurd variability amplitudes or parameter combinations excluded by the simulator is not weakly informative; it is badly calibrated to the scientific problem.

Prior predictive checking is particularly valuable in high-dimensional hierarchical models because marginally reasonable priors can combine into unrealistic joint predictions.

### Worked computational demonstration — Prior sensitivity

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
def posterior_for_prior(mu0,tau):
    lp=-0.5*((mu_grid-mu0)/tau)**2
    z=loglike+lp
    p=np.exp(z-z.max())
    return p/np.trapezoid(p,mu_grid)

for mu0,tau,label in [(8,2,'weak'),(7.5,.2,'informative')]:
    plt.plot(mu_grid,posterior_for_prior(mu0,tau),label=label)
plt.legend(); plt.show()

## 3. Posterior summaries and credible intervals


A 95% Bayesian credible interval is a region containing 95% posterior probability conditional on the model and data. This differs conceptually from a frequentist confidence interval, whose probability statement concerns repeated intervals under repeated sampling.

Posterior summaries should respect geometry. For skewed or multimodal posteriors, a mean and standard deviation can obscure important structure. Report intervals, plots, correlations and, when relevant, probabilities of scientifically meaningful regions.

### Worked computational demonstration — Posterior predictive skeleton

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
rng=np.random.default_rng(2026)
p=posterior_for_prior(8,2)
draw_mu=rng.choice(mu_grid,size=500,replace=True,p=p/p.sum())
rep=np.array([rng.normal(m,sigma,size=len(y)).mean() for m in draw_mu])
print('observed mean:',y.mean())
print('posterior predictive mean interval:',np.quantile(rep,[.025,.975]))

## 4. Posterior predictive checking


The posterior predictive distribution is

\[
p(\tilde D\mid D)=\int p(\tilde D\mid\theta)p(\theta\mid D)\,d\theta.
\]

It asks whether replicated data generated under plausible posterior parameters look like the observations. Choose discrepancy measures that are sensitive to scientifically important failures: tails, multimodality, variability amplitudes, extreme redshifts or correlation structure.

A model can estimate a parameter precisely and still fail posterior predictive checks. Precision is not evidence of adequacy.

## 5. Model comparison and sensitivity


Bayesian evidence naturally appears in model probabilities, but Bayes factors can be sensitive to prior volume. For MSc-level practice the priority should be understanding this sensitivity rather than treating evidence as a universal score.

A good workflow is: formulate competing generative models, perform prior predictive checks, compute posterior inference, perform posterior predictive checks, and then examine whether scientific conclusions are stable to reasonable modelling alternatives.

In the binary-SMBH course, Week 4 reuses the Week-3 log-mass model so students can compare MLE uncertainty with posterior uncertainty under two defensible priors.

## Deep dive — Bayesian workflow rather than "Bayes theorem once"

A robust Bayesian analysis can be organised as a cycle:

1. write the generative model;
2. choose priors;
3. simulate from the prior predictive distribution;
4. fit the observed data;
5. inspect computational diagnostics;
6. perform posterior predictive checks;
7. perform sensitivity analysis;
8. revise the model if necessary.

This cycle matters more than whether a sampler is fashionable.

### Prior parameterisation

Suppose the physical scale is positive. A prior \(p(M)\propto 1\) over a large interval and a prior \(p(\log M)\propto1\) imply different beliefs because

\[
p(M)=p(\log M)\left|\frac{d\log M}{dM}\right|
\propto\frac1M.
\]

Thus "flat" has no invariant meaning. Priors should be described on the scale on which scientific knowledge is expressed.

### Posterior predictive discrepancies

A posterior predictive check requires a chosen discrepancy \(T(D)\). Examples for the SMBH population include a high-redshift tail fraction, skewness of log mass, fraction above a mass threshold, or a correlation coefficient. If observed \(T(D)\) lies in the extreme tail of replicated \(T(D_{\rm rep})\), the model fails to reproduce that aspect of the data.

This is model criticism, not a universal hypothesis test. Different discrepancies probe different failures.

### Precision versus correctness

With 100,000 simulated systems, a simple misspecified model can produce an extremely narrow posterior. The data can tell us precisely which member of the wrong model family best approximates the sample. A narrow posterior is therefore not evidence that the model is physically correct. Posterior predictive checking is essential.

## Binary-SMBH synthesis questions

Answer verbally before leaving the lecture.

1. Which quantity in today's topic is a **property of the simulator**, and which could be an **observable property of a real survey**?
2. What uncertainty or selection effect would most strongly distort the result?
3. What diagnostic would you require before trusting the inference?
4. What information would be unavailable if these were real LSST sources rather than simulations?

## Key equations / ideas to retain

Create your own 6–10 line summary here during class.

## Further reading

- Crete `06_Bayesian` notebooks.
- Gelman et al., *Bayesian Data Analysis*.
- Gabry et al. on visualization in Bayesian workflow.

# Part II — Extended lecture development

## 13. Conjugacy as a transparent update

For

\[
k\sim{\rm Binomial}(n,p),
\qquad
p\sim{\rm Beta}(\alpha,\beta),
\]

the posterior is

\[
p\mid k,n\sim{\rm Beta}(\alpha+k,\beta+n-k).
\]

This makes prior-to-posterior updating explicit and provides a useful check
against numerical methods. Conjugacy is pedagogically useful even though most
real astronomical models are not conjugate.

In [ ]:
grid=np.linspace(0,1,1000)
n,k=20,3
from scipy.stats import beta
for a,b,label in [(1,1,"uniform"),(.5,.5,"Jeffreys-like"),(5,5,"concentrated")]:
    post=beta.pdf(grid,a+k,b+n-k)
    post/=np.trapezoid(post,grid)
    plt.plot(grid,post,label=label)
plt.xlabel("event probability"); plt.ylabel("posterior density")
plt.legend(); plt.show()

## 14. Equal-tailed and highest-density intervals

Equal-tailed intervals use posterior quantiles. Highest-density regions contain
the most probable density values until the required mass is reached. They are
similar for symmetric unimodal posteriors but differ for skewed or multimodal
distributions.

Always specify the interval definition rather than writing an unexplained
"95% error."

## 15. Model posterior odds

Evidence gives a Bayes factor,

\[
B_{10}=\frac{p(D\mid M_1)}{p(D\mid M_0)}.
\]

Posterior model odds additionally require prior odds:

\[
\frac{p(M_1\mid D)}{p(M_0\mid D)}
=
B_{10}\frac{p(M_1)}{p(M_0)}.
\]

A Bayes factor is therefore not itself a posterior model probability.

## 16. Bayesian decision summaries

If an action \(a\) has loss \(L(a,\theta)\), choose

\[
a^\ast=\arg\min_a E[L(a,\theta)\mid D].
\]

Squared-error loss yields the posterior mean; absolute-error loss yields the
posterior median. Familiar summaries therefore encode decision assumptions.

This is the conceptual bridge to Week 11 target selection.

## 17. Posterior predictive discrepancies

A posterior predictive tail area

\[
P[T(D_{\rm rep})\ge T(D_{\rm obs})\mid D]
\]

resembles a p-value but has different calibration because the data help fit the
posterior and are then compared to posterior replications.

Use it as a model-criticism diagnostic, not as a mechanical threshold.

## 18. Analysis priors versus undocumented simulation priors

The supplied files document empirical parameter values/ranges but not every
analytic sampling law. The course therefore does not invent provider priors.

Students may choose explicit **analysis priors** for simplified inference, test
sensitivity to them and label them as analysis choices. This is distinct from
claiming knowledge of how the simulation group sampled the full dataset.

## 19. A complete Bayesian result

A strong report contains likelihood, priors, prior predictive checks,
computational diagnostics, posterior summaries, posterior correlations,
posterior predictive checks, sensitivity analysis and an explicit scientific
limitation.

"Ran MCMC with 50 walkers" is an implementation detail, not a Bayesian
scientific argument.

# Part III — Worked Bayesian reasoning

## Problem A — prior influence as effective information

In the beta-binomial model,

\[
p\sim{\rm Beta}(\alpha,\beta),\qquad
k\sim{\rm Binomial}(n,p),
\]

the posterior is

\[
{\rm Beta}(\alpha+k,\beta+n-k).
\]

Compare two situations:

- \(n=10\), prior equivalent to roughly 20 pseudo-observations;
- \(n=10,000\), the same prior.

The prior can strongly influence the first posterior and have negligible effect
on the second. "Bayesian results depend on priors" is therefore incomplete; the
relevant question is **how much information the data contain relative to the
prior**.

## Problem B — posterior predictive failure despite precise parameters

Imagine \(10^5\) log-mass values are fit with one Gaussian. The posterior for
\((\mu,\sigma)\) can be extremely narrow because of the huge sample.

Yet the observed distribution may be bimodal.

Question: is the posterior "wrong"? Computationally it may be an excellent
posterior **for the wrong model family**. A posterior predictive histogram will
show that replicated one-Gaussian populations fail to reproduce bimodality.

This example separates parameter uncertainty from model uncertainty.

## Problem C — probability of a scientifically relevant region

Instead of reporting only a posterior mean for \(q\), a scientific question may
be

\[
P(q>0.8\mid D).
\]

Posterior samples make this simple:

\[
\widehat P(q>0.8\mid D)
=
\frac1S\sum_s I(q^{(s)}>0.8).
\]

Bayesian posteriors support direct probability statements about parameter
regions, conditional on the model and prior. Ask students to propose one
scientifically meaningful probability statement for each of mass, period and
eccentricity.

In [ ]:
# Posterior-region probability from arbitrary sample array
# example only: a beta posterior
samples=stats.beta.rvs(8,4,size=100_000,random_state=rng)
print("P(q > 0.8 | model,data) ~",np.mean(samples>.8))

# Part IV — Additional Bayesian seminar

## Likelihood dominance is not universal

Students often hear that "with enough data, the prior does not matter." Under
regular identifiable models this can be approximately true for well-constrained
parameters. But priors can remain important when:

- the likelihood is weak or flat;
- parameters lie near boundaries;
- the model is non-identifiable;
- rare-event probabilities are estimated;
- hierarchical scales are weakly constrained;
- model evidence is compared.

Thus prior sensitivity should be tested where the data are weak, not assumed
away because the catalogue is large.

## Model averaging

If several models \(M_k\) remain plausible, predictions can average over model
uncertainty:

\[
p(\widetilde D\mid D)
=
\sum_k
p(\widetilde D\mid D,M_k)
p(M_k\mid D).
\]

This propagates uncertainty about the model choice itself. In practice, model
averaging requires credible candidate models and defensible model priors; it
does not rescue a set in which all models omit essential physics.

### Discussion

Ask students whether a DRW-only and DHO-only variability analysis should be
collapsed into one selected "winner" or whether uncertainty over stochastic
model family might need propagation when testing periodicity.